# NB2 · Hip loads and mass properties

How does leverage change with posture? Compare external load alone, gravity of
the modeled bodies and an example speed-dependent actuator envelope. This study
uses repository geometry and explicit synthetic mass and motor inputs.
[DUK v3 chapter 03](https://app.notion.com/p/4131f3cc03328196a898faaeb9af6530).

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from duk_core.modeling import load_leg_geometry_json

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'robot/config/physical/duk_leg_geometry.json').is_file())
geometry_path = repo / 'robot/config/physical/duk_leg_geometry.json'
fixture_path = repo / 'research/experiments/duk-v3/synthetic.json'
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture['schema_version'] == 1 and fixture['status'] == 'synthetic_research_only'
print('Working tree dirty:', bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo)))
print('Commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture['source'])

In [ ]:
from duk_core.modeling import build_planar_biped_model, leg_jacobian, solve_leg_pose
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.hip_load import LinearTorqueSpeed, ideal_symmetric_load_kg

model = NumericPlanarBiped(build_planar_biped_model(geometry), fixture['planar'])
settings = fixture['hip_envelope']
motor = LinearTorqueSpeed(settings['stall_torque_N_m'], settings['no_load_speed_rad_s'], settings['efficiency'])
angles = np.linspace(75, 130, 56)
q_samples = np.deg2rad(angles)
gravity = model.parameters['g_m_s2']
force = model.masses.sum() * gravity / 2
axle_load = [[0., force], [0., force]]
lever = np.array([leg_jacobian(geometry, solve_leg_pose(geometry, q)).d_o[1] / 1000 for q in q_samples])
effort = np.array([model.static_hip_effort([0, 0, 0, q, q, 0, 0], axle_load) for q in q_samples])
equivalent = [model.equivalent_upper_body(q) for q in q_samples]
payload = [ideal_symmetric_load_kg(j, motor.joint_limit(0), gravity) for j in lever]
test_speed = 2.0
at_speed = motor.joint_limit(test_speed)
saturated = angles[np.max(abs(effort), axis=1) > at_speed]

np.testing.assert_allclose(10 * lever[[15, 45]], [1.333, 2.769], atol=0.0005, rtol=0)
np.testing.assert_allclose(effort[:, 0], effort[:, 1], atol=1e-10, rtol=0)
assert all(e.length_m > 0 and e.inertia_about_center_kg_m2 > 0 for e in equivalent)
print('Synthetic near-static poses exceeding the illustrative envelope at 2 rad/s:', saturated)
print('Upper mass (kg):', equivalent[0].mass_kg)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
axes[0, 0].plot(angles, force * lever, label='Axle load contribution magnitude')
axes[0, 0].plot(angles, abs(effort[:, 0]), label='Required effort with body and link gravity')
axes[0, 0].axhline(at_speed, color='k', linestyle='--', label='Illustrative bound at 2 rad/s')
axes[0, 0].set_ylabel('Joint moment (N·m)')
axes[0, 0].legend()
axes[0, 1].plot(angles, payload, label='Ideal static payload (kg)')
axes[0, 1].plot(angles, abs(lever) * motor.no_load_speed, label='No-load height speed (m/s)')
axes[0, 1].legend()
axes[1, 0].plot(angles, [e.center_mm[0] for e in equivalent], label='Upper CoM x (mm)')
axes[1, 0].plot(angles, [e.center_mm[1] for e in equivalent], label='Upper CoM z (mm)')
axes[1, 0].legend()
axes[1, 1].plot(angles, [e.length_m for e in equivalent], label='Axle–CoM distance (m)')
axes[1, 1].plot(angles, [e.inertia_about_center_kg_m2 for e in equivalent], label='CoM inertia (kg·m²)')
axes[1, 1].legend()
for ax in axes.flat:
    ax.set_xlabel('Linkage angle (deg)')
plt.show()

**Gate.** The two 10 N leverage examples match the v3 reference, the symmetric
effort check passes and the upper-body reduction produces positive mass
properties. The speed-envelope overlay uses the static torque demand to screen
postures. NB5 computes the dynamic joint effort along a moving trajectory.